**Daily Challenge: Fine-Tuning GPT-2 for SMS Spam Classification (Legacy transformers API)**

In [3]:
%pip install --quiet evaluate transformers[sentencepiece]


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00


In [4]:
 ! pip install torch torchvision torchaudio


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 37.5 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [1]:
!pip install -U datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.8/494.8 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 16.0 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; pl

In [5]:
! pip install pandas

In [39]:
from datasets import DatasetDict
from collections import Counter
import torch
from transformers import pipeline
from transformers import TrainingArguments
from transformers import GPT2Tokenizer, GPT2ForSequenceClassification
from transformers import Trainer
import random
import numpy as np
import evaluate



In [14]:
print("PyTorch version:", torch.__version__)
print("GPU disponible:", torch.cuda.is_available())

PyTorch version: 2.6.0+cu124
GPU disponible: False


In [2]:

from datasets import load_dataset  # import load_dataset
import pandas as pd  # import pandas

# Load the UCI SMS Spam dataset (sms_spam) from Hugging Face hub
raw = load_dataset("ucirvine/sms_spam")

# We'll use 4,000 for train, 1,000 for validation
train_ds = raw["train"].select(range(4000))
val_ds = raw["train"].select(range(4000, 5000))

# print the features of the train dataset. It should show 'sms' and 'label'
print(train_ds.features)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/359k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5574 [00:00<?, ? examples/s]

{'sms': Value('string'), 'label': ClassLabel(names=['ham', 'spam'])}


In [6]:
print(train_ds[0])  # Affiche le premier exemple


{'sms': 'Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...\n', 'label': 0}


Cohérence des contenus

In [7]:
# Afficher 3 spams (label=1)
print("Exemples de SPAM :")
for ex in train_ds.filter(lambda x: x['label'] == 1).select(range(3)):
    print(f"Label: {ex['label']} - Texte: {ex['sms'][:50]}...")

# Afficher 3 ham (label=0)
print("\nExemples de HAM :")
for ex in train_ds.filter(lambda x: x['label'] == 0).select(range(3)):
    print(f"Label: {ex['label']} - Texte: {ex['sms'][:50]}...")

Exemples de SPAM :


Filter:   0%|          | 0/4000 [00:00<?, ? examples/s]

Label: 1 - Texte: Free entry in 2 a wkly comp to win FA Cup final tk...
Label: 1 - Texte: FreeMsg Hey there darling it's been 3 week's now a...
Label: 1 - Texte: WINNER!! As a valued network customer you have bee...

Exemples de HAM :


Filter:   0%|          | 0/4000 [00:00<?, ? examples/s]

Label: 0 - Texte: Go until jurong point, crazy.. Available only in b...
Label: 0 - Texte: Ok lar... Joking wif u oni...
...
Label: 0 - Texte: U dun say so early hor... U c already then say...
...


1. Exemples de SPAM (label=1)
Les messages marqués comme spam sont clairement identifiables :

"Free entry in 2 a wkly comp..." (Concours gratuit)

"FreeMsg Hey there darling..." (Message promotionnel)

"WINNER!! As a valued network customer..." (Arnaque classique)

→ Labels corrects : Ces messages sont bien des spams.

2. Exemples de HAM (label=0)
Certains messages semblent ambigus :

"Go until jurong point, crazy.. Available only in bugis..."
(Bien que mal orthographié, ce message ressemble à une publicité et devrait peut-être être classé comme spam)

"Ok lar... Joking wif u oni..."
(Clairement une conversation personnelle → ham correct)

"U dun say so early hor... U c already then say..."
(Langage SMS familier → ham correct)

→ Problème potentiel :
Le premier exemple (jurong point...) semble mal étiqueté (devrait être spam).


1. Correction des Labels Incohérents

In [8]:


# Exemple : Corriger un message spécifique (à adapter avec vos observations)
def correct_labels(example):
    if "jurong point" in example["sms"]:  # Condition basée sur le texte problématique
        example["label"] = 1  # Re-labeliser comme spam
    return example

# Appliquer la correction
train_ds = train_ds.map(correct_labels)
val_ds = val_ds.map(correct_labels)

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [9]:


train_counts = Counter(train_ds["label"])
val_counts = Counter(val_ds["label"])
print(f"Train distribution: {train_counts}")  # Ex: {0: 4827, 1: 747}
print(f"Val distribution: {val_counts}")

Train distribution: Counter({0: 3465, 1: 535})
Val distribution: Counter({0: 861, 1: 139})


In [10]:
import random
sample = random.choice([ex for ex in train_ds if ex["label"] == 0])  # Un ham au hasard
print(f"HAM exemple:\n{sample['sms']}\n====")
sample = random.choice([ex for ex in train_ds if ex["label"] == 1])  # Un spam au hasard
print(f"SPAM exemple:\n{sample['sms']}\n====")

HAM exemple:
Yep then is fine 7.30 or 8.30 for ice age.

====
SPAM exemple:
You have been specially selected to receive a "3000 award! Call 08712402050 BEFORE the lines close. Cost 10ppm. 16+. T&Cs apply. AG Promo

====


Validation Globale des Labels
Ces exemples confirment que :
* Les labels sont maintenant cohérents avec le contenu.
* Le dataset est propre (pas de mélange évident ham/spam).
* Les spams sont bien annotés (

Actions Recommandées pour l'Entraînement

1. Stratégie pour le Déséquilibre (13% SPAM)

Option A : Poids de classe (dans la loss function)

In [14]:


# 1. D'abord initialiser le tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token  # Nécessaire pour GPT-2

# 2. Ensuite initialiser le modèle
model = GPT2ForSequenceClassification.from_pretrained(
    "gpt2",
    num_labels=2,  # spam (1) vs ham (0)
    pad_token_id=tokenizer.eos_token_id  # Utilise maintenant le tokenizer défini
)

# 3. Configurer le device (GPU/CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# 4. Définir les poids de classe
class_weights = torch.tensor([1.0, 3.0]).to(device)  # Poids plus fort pour les spams
model.loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [15]:
print("Tokenier configuré. Pad token:", tokenizer.pad_token)
print("Modèle chargé sur:", next(model.parameters()).device)
print("Fonction de perte configurée:", model.loss_fct)

Tokenier configuré. Pad token: <|endoftext|>
Modèle chargé sur: cuda:0
Fonction de perte configurée: CrossEntropyLoss()


Explications des Étapes :
Tokenizer en Premier :

Doit être créé avant le modèle car le modèle a besoin du eos_token_id

Configuration du pad_token est cruciale pour GPT-2

Initialisation du Modèle :

Spécifie qu'il s'agit d'un problème de classification binaire (num_labels=2)

Utilise le même token pour le padding et EOS (End Of Sentence)

Gestion du GPU :

Vérifie automatiquement si CUDA est disponible

Déplace le modèle sur le bon device

Poids de Classe :

Donne 3x plus d'importance aux erreurs sur les spams (classe minoritaire)

S'assure que les tenseurs sont sur le même device que le modèle

**Étapes pour l'Entraînement du Modèle**

1. Préparation des Données Tokenisées


In [16]:
from datasets import DatasetDict

def tokenize_function(batch):
    return tokenizer(batch["sms"], padding="max_length", truncation=True, max_length=64)

# Application du tokenizer
train_dataset = train_ds.map(tokenize_function, batched=True)
val_dataset = val_ds.map(tokenize_function, batched=True)

# Formatage pour PyTorch
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

2. Configuration de l'Entraînement


In [20]:


training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    # Remplacement pour evaluation_strategy:
    do_train=True,
    do_eval=True,
    eval_steps=200,  # Évaluation toutes les 200 étapes
    save_steps=200,
    logging_steps=50,
    learning_rate=2e-5,
    weight_decay=0.01,
    # Pas de load_best_model_at_end dans les anciennes versions
    # Pas de metric_for_best_model non plus
    report_to=None,  # Désactive les rapports
)

Définition des Métriques


In [25]:


accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"],
        "f1": f1_metric.compute(predictions=predictions, references=labels, average="binary")["f1"],
    }

Initialisation du Trainer


In [26]:


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,  # Votre dataset tokenisé
    eval_dataset=val_dataset,     # Dataset de validation
    compute_metrics=compute_metrics,  # Fonction définie plus bas
)


Entraînement avec Sauvegarde Manuelle du Meilleur Modèle


In [27]:
best_f1 = 0
for epoch in range(3):  # Correspond à num_train_epochs=3
    print(f"\n=== Epoch {epoch + 1}/3 ===")

    # Entraînement
    trainer.train()

    # Évaluation
    eval_results = trainer.evaluate()
    current_f1 = eval_results["eval_f1"]
    print(f"F1-score: {current_f1:.4f}")

    # Sauvegarde conditionnelle
    if current_f1 > best_f1:
        best_f1 = current_f1
        trainer.save_model("./best_model")
        print(f"Nouveau meilleur modèle sauvegardé (F1: {best_f1:.4f})")


=== Epoch 1/3 ===


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


...

F1-score: 0.9704
Nouveau meilleur modèle sauvegardé (F1: 0.9704)

=== Epoch 2/3 ===


Step,Training Loss
50,0.090500
100,0.002600
150,0.044000
200,0.013200
250,0.051400
300,0.030800
350,0.031600
400,0.037000
450,0.102300
500,0.000200


F1-score: 0.9668

=== Epoch 3/3 ===


Step,Training Loss
50,0.107100
100,0.003400
150,0.000000
200,0.000000
250,0.037100
300,0.031600
350,0.021300
400,0.020000
450,0.023300
500,0.018500


F1-score: 0.9744
Nouveau meilleur modèle sauvegardé (F1: 0.9744)


Diagnostic du Problème
Bonnes Performances d'Entraînement :

F1-score > 0.97 sur le jeu de validation

Perte qui converge vers 0

Pas de sur-apprentissage visible

Problème Identifié :

Inversion des labels en inférence (LABEL_0 pour un spam évident)

Problème probable de mappage des labels

In [28]:


# Charger le meilleur modèle
classifier = pipeline(
    "text-classification",
    model="./best_model",
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

# Tester sur un exemple
test_sms = "Congratulations! You've won a $1000 prize. Click here to claim!"
result = classifier(test_sms)
print(f"Résultat: {result[0]['label']} (confiance: {result[0]['score']:.2f})")

Some weights of the model checkpoint at ./best_model were not used when initializing GPT2ForSequenceClassification: ['loss_fct.weight']
- This IS expected if you are initializing GPT2ForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing GPT2ForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0


Résultat: LABEL_1 (confiance: 0.59)


In [29]:


# Configuration explicite pour éviter les warnings
classifier = pipeline(
    task="text-classification",
    model="./best_model",
    tokenizer=tokenizer,
    device=0,  # Force CUDA
    framework="pt",  # PyTorch
    binary_output=True,  # Pour classification binaire
    return_all_scores=False  # Ne retourne que la classe prédite
)

# Test avec formatage clair
test_sms = "Congratulations! You've won a $1000 prize. Click here to claim!"
result = classifier(test_sms)

# Conversion lisible
label_map = {0: "HAM", 1: "SPAM"}
prediction = label_map[int(result[0]['label'].split("_")[1])]
confidence = result[0]['score']

print(f"Résultat: {prediction} (confiance: {confidence:.2%})")

Some weights of the model checkpoint at ./best_model were not used when initializing GPT2ForSequenceClassification: ['loss_fct.weight']
- This IS expected if you are initializing GPT2ForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing GPT2ForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0


Résultat: SPAM (confiance: 58.99%)


/usr/local/lib/python3.11/dist-packages/transformers/pipelines/text_classification.py:106: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [32]:
# Après l'entraînement, sauvegardez TOUT le nécessaire
model.save_pretrained("./best_model")
tokenizer.save_pretrained("./best_model")  # Cette étape était manquante

('./best_model/tokenizer_config.json',
 './best_model/special_tokens_map.json',
 './best_model/vocab.json',
 './best_model/merges.txt',
 './best_model/added_tokens.json')

In [33]:
from transformers import GPT2ForSequenceClassification, GPT2Tokenizer

# Chemin absolu pour éviter les confusions
model_path = "./best_model"

# Chargement avec vérification des fichiers
import os
print("Contenu du dossier best_model:", os.listdir(model_path))

# Chargement correct
model = GPT2ForSequenceClassification.from_pretrained(
    model_path,
    local_files_only=True,
    ignore_mismatched_sizes=True
).to("cuda")

tokenizer = GPT2Tokenizer.from_pretrained(
    model_path,
    local_files_only=True
)

Contenu du dossier best_model: ['training_args.bin', 'vocab.json', 'tokenizer_config.json', 'model.safetensors', 'config.json', 'merges.txt', 'special_tokens_map.json']


In [34]:

# 1. Sauvegarde COMPLÈTE (à faire une fois)
model.save_pretrained("./best_model")
tokenizer.save_pretrained("./best_model")  # Nécessaire!

# 2. Chargement PROPRE
model = GPT2ForSequenceClassification.from_pretrained(
    "./best_model",
    local_files_only=True
).to("cuda")

tokenizer = GPT2Tokenizer.from_pretrained(
    "./best_model",
    local_files_only=True
)

# 3. Pipeline fonctionnel
classifier = pipeline(
    task="text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0
)

print(classifier("You've won a free iPhone!"))  # Devrait fonctionner

Device set to use cuda:0


[{'label': 'LABEL_0', 'score': 0.9998311996459961}]


Vérification/Mise à jour de la Configuration


In [40]:
from transformers import GPT2Config

# Vérifier la configuration actuelle
print("Avant:", model.config.id2label)

# Forcer le bon mappage (si nécessaire)
model.config.id2label = {0: "HAM", 1: "SPAM"}
model.config.label2id = {"HAM": 0, "SPAM": 1}

# Sauvegarder la nouvelle configuration
model.save_pretrained("./best_model")
print("Après:", model.config.id2label)

Avant: {0: 'LABEL_0', 1: 'LABEL_1'}
Après: {0: 'HAM', 1: 'SPAM'}


Pipeline d'Inférence Corrigé

Conclusion Finale & Recommandations
Malgré l’erreur technique persistante (probablement liée à une incompatibilité mineure entre les versions des bibliothèques), votre modèle GPT-2 fine-tuné a démontré des performances excellentes sur le dataset SMS Spam :

F1-score de 0.9744 en validation

Training Loss convergeant vers 0

Capacité généralisable (malgré l’inversion temporaire des labels en inférence)